In [1]:
import re
import numpy as np
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tabulate import tabulate
from collections import Counter


In [2]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [3]:
def preprocess_text(text):
    text = text.lower()                         # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # remove punctuation/numbers
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS]  # remove stopwords
    return words

In [4]:
tokenized_dataset = [preprocess_text(doc) for doc in dataset]

print("Preprocessed Tokenized Dataset:")
for doc in tokenized_dataset:
    print(doc)

Preprocessed Tokenized Dataset:
['love', 'playing', 'football', 'weekends']
['enjoy', 'hiking', 'camping', 'mountains']
['like', 'read', 'books', 'watch', 'movies']
['prefer', 'playing', 'video', 'games', 'sports']
['love', 'listening', 'music', 'going', 'concerts']


In [5]:
word2vec_model = Word2Vec(
    sentences=tokenized_dataset,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [6]:
X = np.array([
    np.mean([word2vec_model.wv[word] for word in doc if word in word2vec_model.wv], axis=0)
    for doc in tokenized_dataset
])

In [7]:
k = 2
km = KMeans(n_clusters=k, random_state=42)
km.fit(X)

y_pred = km.predict(X)

C:\Users\user\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [8]:
processed_docs_as_text = [" ".join(doc) for doc in tokenized_dataset]

table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(processed_docs_as_text, y_pred)])
print("\nWord2Vec Clustering Results:")
print(tabulate(table_data, headers="firstrow"))


Word2Vec Clustering Results:
Document                               Predicted Cluster
-----------------------------------  -------------------
love playing football weekends                         0
enjoy hiking camping mountains                         0
like read books watch movies                           1
prefer playing video games sports                      0
love listening music going concerts                    1


In [9]:
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Purity: 0.6


**Do the Purity differ when applying text preprocessing before vectorization?**

Yes, the purity may differ after applying text preprocessing before vectorization.
This is because preprocessing removes noisy words such as stopwords, punctuation, and unnecessary variations, so the vectorizer focuses more on meaningful terms. As a result, the document representation becomes cleaner, which can change the clustering result and therefore the purity. However, the purity does not always improve; it depends on the dataset and how much useful information is removed during preprocessing. The lab also explains that preprocessing helps reduce noise and makes the dataset easier to work with before feature extraction and clustering.